# 05 · Whole-slide inference and dual-head evaluation

Read trained weights and real slide inputs, predict whole-slide annotations, and save prediction maps and validation diagnostics.

## 1. Read the input data

Edit the real input paths and method parameters below. Run the cells in order using the `spagc` kernel. Each notebook reads its own inputs; it does not execute another notebook.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks/02_he_prediction").is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Start Jupyter inside the spaGC repository.")
    ROOT = ROOT.parent


if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_ID = os.environ.get("SPAGC_RUN_ID", "he_reference_v2")
OUT = ROOT / "outputs/he_prediction" / RUN_ID
DEVICE = os.environ.get(
    "SPAGC_DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu"
)
FEATURE_DIR = OUT / "features"
SLIDE_ROWS = [
    ("proseg1", 984, 1784, 27761, 50000, "proseg1.tif", "proseg1.h5"),
    ("proseg2", 824, 1784, 23140, 50000, "proseg2.tif", "proseg2.h5"),
    ("proseg3", 856, 1784, 23981, 50000, "proseg3.tif", "proseg3.h5"),
    ("proseg4", 912, 1784, 25667, 50000, "proseg4.tif", "proseg4.h5"),
    ("proseg5", 896, 1784, 25108, 50000, "proseg5.tif", "proseg5.h5"),
    ("proseg6", 912, 1784, 25706, 50000, "proseg6.tif", "proseg6.h5"),
    ("proseg7", 1624, 3472, 45603, 97266, "proseg7.tif", "proseg7.h5"),
    ("proseg8", 1480, 3472, 41510, 97267, "proseg8.tif", "proseg8.h5"),
    ("proseg9", 1624, 3472, 45603, 97262, "proseg_9.tif", "proseg_9.h5"),
    ("proseg10", 1624, 3472, 45605, 97259, "proseg_10.tif", "proseg_10.h5"),
]
SLIDES = []
for (
    name,
    height,
    width,
    native_height,
    native_width,
    image_name,
    feature_name,
) in SLIDE_ROWS:
    index = name.removeprefix("proseg")
    SLIDES.append(
        {
            "name": name,
            "h5ad_path": str(
                ROOT / "data/he_aligned_h5ad" / f"proseg_{index}_aligned.h5ad"
            ),
            "h5_path": str(FEATURE_DIR / feature_name),
            "he_tif_path": str(ROOT / "data/he_slides" / image_name),
            "niche_label_path": str(
                ROOT
                / "data/he_export_2026_10_09/labels"
                / f"Proseg{index}"
                / "label_niche_uint16.dat"
            ),
            "domain_label_path": str(
                ROOT
                / "data/he_export_2026_10_09/labels"
                / f"Proseg{index}"
                / "label_domain_uint16.dat"
            ),
            "label_shape": [height, width],
            "H": height,
            "W": width,
            "level0_shape": [native_height, native_width],
            "he_shape": [height * 28, width * 28],
            "region_size": 28,
        }
    )
TRAIN_SLIDES = SLIDES[:8]
VALIDATION_SLIDES = SLIDES[8:]
SAMPLE_METADATA = os.environ.get("SPAGC_SAMPLE_METADATA")
MODEL = OUT / "training/model_best.pth"
BACKBONE = ROOT / "models/he_prediction/export_2026_10_09/ctranspath.pth"
INFERENCE_SOURCE = "H5"
PRED_DIR = OUT / "predictions"

In [ ]:
from spagc._io import require_files
from spagc.histology.dataset import build_slide_dataset
from spagc.histology.inference import extract_and_infer_multi_slices, infer_multi_slices
from spagc.histology.metadata import (
    attach_sample_metadata,
    prepare_slides,
    setup_slide_shapes,
)
from spagc.histology.metrics import evaluate
from spagc.histology.model import load_segmentation_model

Read the real slide records, their image geometry and the checkpoint produced by training. HDF5 inference requires cached CTransPath features; RAM inference additionally requires pretrained encoder weights.

In [ ]:
TRAIN_SLIDES = attach_sample_metadata(TRAIN_SLIDES, SAMPLE_METADATA)
VALIDATION_SLIDES = attach_sample_metadata(VALIDATION_SLIDES, SAMPLE_METADATA)
require_files(MODEL)
slides = prepare_slides(TRAIN_SLIDES + VALIDATION_SLIDES, root=ROOT)
setup_slide_shapes(slides)
for slide in slides:
    require_files(slide["he_tif_path"])
    if INFERENCE_SOURCE == "H5":
        require_files(slide["h5_path"])

## 2. Predict whole-slide niche and domain maps

Accumulate centre-weighted tile probabilities, take class decisions on the region grid, and apply the inference tissue mask. Stream maps to disk rather than keeping every whole slide in memory.

Refine frozen image tokens, combine them with regional RGB features, and predict separate niche and domain logits.

Use the original convolutional stem in the Swin-based CTransPath architecture.

Build CTransPath and strictly restore the supplied pretrained weights; keep the encoder frozen.

In [ ]:
common = dict(
    model_path=str(MODEL),
    pred_dat_dir=str(PRED_DIR),
    slides=slides,
    device=DEVICE,
    num_domain_classes=5,
    num_niche_classes=21,
    batch_size=128,
    num_workers=0,
    mask_background=not bool(None),
    smooth_sigma=0.0,
    skip_existing=False,
)
if INFERENCE_SOURCE == "H5":
    infer_multi_slices(**common, load_h5_RAM="No")
elif INFERENCE_SOURCE == "RAM":
    require_files(BACKBONE)
    ram_common = {key: value for (key, value) in common.items() if key != "batch_size"}
    extract_and_infer_multi_slices(
        **ram_common,
        ctranspath_checkpoint=str(BACKBONE),
        inference_batch_size=common["batch_size"],
        feature_batch_size=256,
        stride=112,
        slide_level=0,
    )
else:
    raise ValueError("INFERENCE_SOURCE must be H5 or RAM")
for slide in slides:
    for head in ("domain", "niche"):
        prediction = Path(slide[f"{head}_pred_dat_path"])
        assert prediction.stat().st_size == int(np.prod(slide["label_shape"])) * 2
        print(prediction)

## 3. Evaluate the declared validation slides

Compute tile diagnostics on validation slides only, not on the training cohort. These scores are validation diagnostics rather than independent test results; save tile_metrics.json with that scope.

Accumulate foreground confusion matrices and report segmentation metrics without updating the model.

In [ ]:
(model, metadata) = load_segmentation_model(MODEL)
metrics = None
if INFERENCE_SOURCE == "H5" and all(
    Path(s.get("h5_path", "missing")).is_file() for s in slides
):
    evaluation_names = {record["name"] for record in VALIDATION_SLIDES}
    evaluation_slides = [
        record for record in slides if record["name"] in evaluation_names
    ]
    if not evaluation_slides:
        raise ValueError(
            "No declared validation slides are present for tile diagnostics."
        )
    dataset = build_slide_dataset(
        evaluation_slides,
        num_niche_classes=model.niche_head.out_channels,
        load_h5_RAM="No",
        standardize_features=metadata.get("standardize_features", True),
    )
    try:
        metrics = evaluate(
            model.to(DEVICE),
            DataLoader(dataset, batch_size=128),
            DEVICE,
            num_domain_classes=model.domain_head.out_channels,
            num_niche_classes=model.niche_head.out_channels,
        )
        metrics["evaluation_scope"] = "fixed_validation_tile_diagnostic"
        metrics["independent_test"] = False
        (PRED_DIR / "tile_metrics.json").write_text(json.dumps(metrics, indent=2))
        print(metrics)
    finally:
        for child in getattr(dataset, "datasets", [dataset]):
            child.close()
else:
    print(
        "RAM inference or no matched H5: tile diagnostics require cached HDF5 features."
    )